# Specialization and detection under stimulation

Six experiments, in dependency order. Roughly 4 hours of MATLAB in total, all cached, so
you can stop after any section and pick up later.

| § | Experiment | MATLAB |
|---|---|---|
| 2 | Does detection survive stimulation at all? | ~37 min |
| 3 | Train a T3P agent per severity (0.3, 0.6, 1.0) | ~24 min |
| 4 | Sweep every agent across seven brain severities | ~68 min |
| 5 | Detection with a real agent running | ~28 min |
| 6 | Smaller step: 0.5 → 0.6 | ~28 min |
| 7 | Cost of letting the agent re-learn instead | ~60 min |

Section 2 needs no trained agent and tests the assumption the whole paper rests on, so it
goes first. If separation collapses there, sections 3 and 4 still stand but the claim
changes, and better to know on day one.

Run from the workspace root. `experiments.py`, `pd_switch.py`, `dbs_t3p.py` and
`helpers.py` all need to be importable, and MATLAB needs `matlab/` and `gating/`.

## 1. Setup

One MATLAB engine for everything. `bgn_vars.mat` is a single shared state file, so a
second engine would corrupt the first. `dbs_t3p.MatlabBackend` owns it and the
`pd_switch` functions borrow `backend.eng`.

In [ ]:
%matplotlib inline

import sys, os
from pathlib import Path
print(sys.executable)
print(os.getcwd())
import matlab.engine
print("matlab.engine ok")

In [ ]:
import importlib
import numpy as np
import matplotlib.pyplot as plt

import helpers
import dbs_t3p as t3p
import pd_switch as ps
import experiments as ex
for m in (t3p, ps, ex):
    importlib.reload(m)

backend = t3p.MatlabBackend(code_dir=".", parent_dir="matlab")
backend.eng.addpath(str(Path("gating").resolve()), nargout=0)
eng = backend.eng
print("engine up, gating on path")

`MatlabBackend` looks for gating at `matlab/gating`, which does not exist here — yours is
at the repo root. The explicit `addpath` above fixes the warning it prints and makes sure
the gating functions are actually reachable.

## 2. Does detection survive stimulation?

Every detection result so far used `amp = 0`. The paper claims we can spot a severity
change *while a controller is running*, so this is the assumption to test first.

No trained agent needed: a T3P agent's output is a fixed frequency and amplitude, so
"stimulation is running" just means a non-zero `amp` passed to the usual switch run.

10 seeds at each of four amplitudes, about 37 minutes.

In [ ]:
stim_sweep = {}
for amp in (0.0, 250.0, 1000.0, 3500.0):
    res, _ = ps.run_seeds(eng, range(1, 11), helpers.calculate_pb,
                          out_dir=f"runs_stim_a{int(amp)}", freq=130.0, amp=amp)
    stim_sweep[amp] = res
    ex.report_detection(f"amp = {amp:.0f} uA", res)

What to read. `gap` is how much P_β rises across the switch, `separation` is that rise in
units of within-state noise, `worst` is the hardest of the ten brains, and `detected` is
how many the changepoint test caught at the current threshold.

If separation holds near its unstimulated value out to realistic amplitudes, the full-loop
claim is safe. If it collapses at 250 µA, the paper becomes "specialized controllers work,
and here is the measured limit on detecting a switch while they run" — still a result,
just a different one, and worth knowing now rather than in three weeks.

130 Hz is used here as a stand-in. Section 5 repeats this with the real agent's setting.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
amps = sorted(stim_sweep)
means = [ps.summarize(stim_sweep[a])[1].mean() for a in amps]
worst = [abs(ps.summarize(stim_sweep[a])[1]).min() for a in amps]
ax.plot(amps, means, marker="o", lw=2, label="mean over 10 brains")
ax.plot(amps, worst, marker="s", lw=1.5, ls="--", label="worst brain")
ax.axhline(3, color="tab:green", ls=":", lw=1.2)
ax.text(amps[0], 3, " clustering reliable above here", va="bottom", fontsize=9,
        color="tab:green")
ax.set_xlabel("stimulation amplitude (uA)")
ax.set_ylabel("separation (sd)")
ax.set_title("Does the severity change stay visible under stimulation?")
ax.legend()
ax.grid(alpha=0.3)
fig

## 3. Train an agent per severity

Three severities: 0.3, 0.6 and 1.0 — the same values the existing reward-model datasets
were collected at. That makes a free cross-check available later: does the bandit converge
on the setting the reward model picks at each severity? Two independent methods agreeing is
worth reporting, and disagreeing is worth more.

T3P over a 49-arm grid: 6 frequencies by 8 non-zero amplitudes, plus a no-stimulation arm.

The default current grid in `dbs_t3p` starts at 1000 µA, which leaves everything below
that unsampled — and the reward-model work found reward falls monotonically with
amplitude, so the interesting region is exactly the one the default grid skips. The fine
grid below is the one from `dbs_rerun.ipynb` cell 42.

In [ ]:
ex.use_fine_grid()

In [ ]:
agents = ex.train_agents(backend, helpers, pd_values=(0.3, 0.6, 1.0), tmax=1000)
print()
flagged = ex.report_agents(agents)

Two failure modes the report checks for automatically.

**All three severities choosing the same arm** would mean a selector has nothing to select
between, and the project's premise is wrong. That is the single most important line of
output in this notebook.

**A grid-edge selection** — top frequency, top amplitude, or no stimulation at all. This
project has produced an apparent optimum sitting on a grid boundary four times now, and an
optimum at the edge usually means the real one is outside the box. If something is flagged,
widen that dimension and retrain before using the agent for anything downstream.

## 4. Cross-controller matrix

Every agent, swept across seven brain severities — including four nobody trained on.
5 seconds per run, 10 seeds, the same seeds for the same brain across agents so the
comparisons are paired.

Testing only at the agents' own severities would tell you the diagonal wins. Sweeping tells
you **how far from its training severity a controller stays competent**, and where two
curves cross is where a selector should switch between them. That is the figure.

It also answers the question a reviewer will ask: how many controllers does the library
actually need? Flat, overlapping curves mean three is too many. A sharp fall two steps from
the training point means three is too few.

About 68 minutes.

In [ ]:
BRAINS = (0.3, 0.4, 0.5, 0.6, 0.7, 0.85, 1.0)

rows = ex.cross_matrix(backend, helpers, agents, brain_pds=BRAINS,
                       seeds=range(1, 11), tmax=5000)
print()
ex.report_matrix(rows)

In [ ]:
fig = ex.plot_matrix(rows)
fig

Each curve should peak near its own training severity, marked by the dotted line in its
colour, and fall away on either side. The paired table is the evidence behind the picture:
for every brain it compares the nearest-trained agent against the others on identical
seeds and says whether the difference clears two standard errors.

Brains at 0.4, 0.5, 0.7 and 0.85 have no agent of their own, so they are matched to the
nearest one — those rows are the interpolation test.

"not separated" everywhere is a negative result and a publishable one — it would say the
controllers are interchangeable at these severities, which kills the selector but is worth
reporting honestly.

In [ ]:
ex.report_matrix(rows, key="pb")

The same matrix on the raw biomarker rather than reward. Worth looking at because reward
mixes P_β with an energy penalty, and the earlier reward models collapsed onto
no-stimulation precisely because the energy term dominated. If the diagonal is strong in
P_β but weak in reward, the reward weighting is the problem rather than the controllers.

## 5. Detection with a real agent running

Section 2 used 130 Hz as a placeholder. This repeats it with the pd = 0.3 agent's actual
setting, which is the deployment case: the patient is on their 0.3 controller, their state
changes to 0.7, and nobody has told the controller yet.

30 seeds, about 28 minutes.

In [ ]:
res_stim = ex.detect_under_stim(eng, helpers, agents[0.3],
                                seeds=range(1, 31), pd_start=0.3, pd_end=0.7)
ex.report_detection("0.3 -> 0.7, agent 0.3 running", res_stim)

fig = ps.plot_seeds(res_stim, out_png="pd_switch_under_stim.png")
fig

Compare this against the unstimulated 30-seed result from `pd_switch.ipynb`: +23.4% gap,
5.10 sd separation, 19/20 held out. The difference between the two is the cost of running
a controller, and it is a figure in the paper either way.

## 6. A smaller step: 0.5 → 0.6

Expect this to fail, and plan to report it as the measured limit of the method.

The worst brain at 0.3 → 0.7 was already 1.67 sd, where any detector deciding from one
window at a time mislabels about one window in five. A 0.5 → 0.6 step is a fraction of
that size. A stated boundary is a result; a boundary a reviewer finds is a problem.

15 seeds rather than 30, because establishing a failure does not need the same precision as
establishing a result. If it unexpectedly works, re-run at 30 before reporting it.

The 0.6 agent runs here rather than the 0.3 one, since 0.6 is the nearest trained
controller to a patient sitting between 0.5 and 0.6.

In [ ]:
res_small, _ = ps.run_seeds(eng, range(1, 16), helpers.calculate_pb,
                            out_dir="runs_switch_05to06",
                            pd_start=0.5, pd_end=0.6, freq=130.0, amp=0.0)
ex.report_detection("0.5 -> 0.6, no stimulation", res_small)

In [ ]:
res_small_stim = ex.detect_under_stim(eng, helpers, agents[0.6],
                                      seeds=range(1, 16), pd_start=0.5, pd_end=0.6)
ex.report_detection("0.5 -> 0.6, agent 0.6 running", res_small_stim)

## 7. What it costs to let the agent re-learn

The alternative to detect-and-switch: keep the bandit live, and when the brain changes let
its own falling rewards push it to new parameters. Two things to measure — how fast it
notices, and what the noticing costs the patient.

The published `T3PBandit` cannot do this, for three reasons, so `AdaptiveT3P` overrides it:

- **`Q += (r - Q) / counts`** averages over every pull ever, so after 60 pulls a new reward
  moves Q by under 2%. Replaced with a constant step size, `Q += alpha * (r - Q)`.
- **Epsilon decays to zero** by round 8, after which no other arm is ever sampled and their
  Q values stay frozen at their pre-change estimates. Replaced with a floor.
- **Pruning is permanent.** The top-25 is chosen on pre-change performance, so the arm that
  is best afterwards may already have been discarded. Disabled.

`dbs_t3p.py` is untouched.

The most expensive section, roughly 20 minutes per seed. Start with 3 and add more only if
the spread is wide.

In [ ]:
TARGET = ex.arm_index(agents[1.0]["freq"], agents[1.0]["curr"])
ORACLE = np.mean([r["reward"] for r in rows
                  if r["agent_pd"] == 1.0 and r["brain_pd"] == 1.0])
print(f"target arm {TARGET} = {t3p.ARMS[TARGET]}, oracle reward {ORACLE:+.4f}")

adaptive = [ex.run_adaptive_switch(backend, helpers, pd_before=0.3, pd_after=1.0,
                                   rounds_before=50, rounds_after=200, tmax=1000,
                                   target_arm=TARGET, oracle_reward=ORACLE,
                                   seed=s, eps_floor=0.30)
            for s in (1, 2, 3)]
print()
ex.report_adaptive(adaptive)

In [ ]:
fig = ex.plot_adaptive(adaptive, oracle_reward=ORACLE)
fig

The two vertical markers are the whole result. The agent drops its old setting almost
immediately — that is a usable change signal, and it is what you meant by measuring a
change in stimulation. Finding *good* new parameters takes far longer, and the gap between
the two lines is stimulation the patient receives while the agent is still searching.

On a toy landscape the agent took around 115 pulls to re-optimise. If the real one is
similar, that is roughly two minutes of brain time on wrong settings — which is the number
that justifies a pre-trained library, and the baseline the detect-and-switch result should
be compared against.

Caveats worth stating in the paper: `eps_floor`, `alpha` and the number of post-change
rounds all affect how fast it recovers, and with too few rounds it never recovers at all.
Sweep them rather than reporting one setting.

## 8. Summary

In [ ]:
print("agents")
ex.report_agents(agents)

print("\n\ndetection")
ex.report_detection("0.3 -> 0.7, no stim", stim_sweep[0.0])
ex.report_detection("0.3 -> 0.7, agent running", res_stim)
ex.report_detection("0.5 -> 0.6, no stim", res_small)
ex.report_detection("0.5 -> 0.6, agent running", res_small_stim)

print("\n\nspecialization")
ex.report_matrix(rows)

print("\n\nletting the agent re-learn instead")
ex.report_adaptive(adaptive)

In [ ]:
backend.close()
print("engine closed")